# Benchmark: WDM Time-Shift Backends

This notebook compares lag-first chunked backends against legacy assembly on laptop-safe sizes.
Run using the `lisatools` conda environment. vmap is intentionally disabled by default.

In [8]:
# --- Cell 1: imports and environment ---

import os
import sys
import csv
import time
from datetime import datetime

import numpy as np

import jax
import jax.numpy as jnp

from few.utils.constants import YRSID_SI
from few.waveform import GenerateEMRIWaveform


from lisaorbits.file import read_orbit_file
from lisaorbits import InterpolatedOrbits

# Adjust these imports only if your local package paths differ.
from WDM_LISAresponse.code.lisa_response.waveforms import generate_windowed_padded_waveform
from WDM_LISAresponse.code.lisa_response.response import (
    one_arm_response_td,
    one_arm_response_wdm_dT,
)
from WDM_LISAresponse.code.lisa_response.TDI import (
    gen2_michelson_wdm,
    gen2_michelson_td,
)

from WDM.code.time_delay_filters import time_shift_fast as tsf
import WDM

# Context-reused one-arm benchmarking helpers
from WDM_LISAresponse.code.lisa_response.response_sub import build_sky_basis
from WDM_LISAresponse.code.lisa_response.response import (
    _build_wdm_dT_context,
    _one_arm_response_wdm_dT_from_context,
)

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("JAX devices:", jax.devices())
print("Current working directory:", os.getcwd())
print("Run this notebook from the lisatools conda environment.")

Python executable: /Users/tomekkinowski/opt/anaconda3/envs/lisatools/bin/python
Python version: 3.13.11 | packaged by conda-forge | (main, Dec  6 2025, 11:37:04) [Clang 19.1.7 ]
JAX devices: [CpuDevice(id=0)]
Current working directory: /Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse/examples
Run this notebook from the lisatools conda environment.


In [2]:
# --- Cell 2: backend configurations ---

# Exact/reference backends.
# Keep empty for routine laptop sweeps because exact c128 is very expensive
# at large Nt / small Nf.
REFERENCE_BACKENDS = {}

# Exact Tl/Tp but complex64 assembly.
# Optional; currently not the main focus.
EXACT_FAST_BACKENDS = {}

# Fast approximate interpolation sweep.
# These compare different Tl/Tp interpolation grid sizes.
INTERP_BACKENDS = {
    "chunked_c64_R128_interp4": {
        "assembly_backend": "lagfirst_chunked",
        "assembly_precision": "complex64",
        "row_chunk_size": 128,
        "tl_tp_mode": "interp",
        "tl_tp_interp_points": 4,
        "tl_tp_interp_kind": "linear",
    },
    "chunked_c64_R128_interp8": {
        "assembly_backend": "lagfirst_chunked",
        "assembly_precision": "complex64",
        "row_chunk_size": 128,
        "tl_tp_mode": "interp",
        "tl_tp_interp_points": 8,
        "tl_tp_interp_kind": "linear",
    },
    "chunked_c64_R128_interp16": {
        "assembly_backend": "lagfirst_chunked",
        "assembly_precision": "complex64",
        "row_chunk_size": 128,
        "tl_tp_mode": "interp",
        "tl_tp_interp_points": 16,
        "tl_tp_interp_kind": "linear",
    },
    "chunked_c64_R128_interp32": {
        "assembly_backend": "lagfirst_chunked",
        "assembly_precision": "complex64",
        "row_chunk_size": 128,
        "tl_tp_mode": "interp",
        "tl_tp_interp_points": 32,
        "tl_tp_interp_kind": "linear",
    },
     "chunked_c64_R128_interp64": {
        "assembly_backend": "lagfirst_chunked",
        "assembly_precision": "complex64",
        "row_chunk_size": 128,
        "tl_tp_mode": "interp",
        "tl_tp_interp_points": 64,
        "tl_tp_interp_kind": "linear",
    },
}

# Legacy route retained for regression only.
# Do not run on laptop unless intentionally comparing old paths.
LEGACY_BACKENDS = {}

# -------------------------------------------------------------------
# Laptop-safe run switches
# -------------------------------------------------------------------

RUN_REFERENCE = False
RUN_EXACT_FAST = False
RUN_INTERP = True
RUN_LEGACY = False
RUN_TDI = False


def selected_backends():
    """Return active backend dictionary for this notebook run."""
    out = {}

    if RUN_LEGACY:
        out.update(LEGACY_BACKENDS)

    if RUN_REFERENCE:
        out.update(REFERENCE_BACKENDS)

    if RUN_EXACT_FAST:
        out.update(EXACT_FAST_BACKENDS)

    if RUN_INTERP:
        out.update(INTERP_BACKENDS)

    return out


BACKENDS = selected_backends()

print("Active backend configurations:")
for name, kwargs in BACKENDS.items():
    print(f"  {name}: {kwargs}")

print()
print("RUN_REFERENCE:", RUN_REFERENCE)
print("RUN_EXACT_FAST:", RUN_EXACT_FAST)
print("RUN_INTERP:", RUN_INTERP)
print("RUN_LEGACY:", RUN_LEGACY)
print("RUN_TDI:", RUN_TDI)

if not RUN_REFERENCE:
    print()
    print(
        "NOTE: exact c128 reference is disabled. "
        "TD-reference columns remain meaningful, but "
        "rel_l2_vs_faithful is only a fast-mode consistency diagnostic."
    )

Active backend configurations:
  chunked_c64_R128_interp4: {'assembly_backend': 'lagfirst_chunked', 'assembly_precision': 'complex64', 'row_chunk_size': 128, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 4, 'tl_tp_interp_kind': 'linear'}
  chunked_c64_R128_interp8: {'assembly_backend': 'lagfirst_chunked', 'assembly_precision': 'complex64', 'row_chunk_size': 128, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 8, 'tl_tp_interp_kind': 'linear'}
  chunked_c64_R128_interp16: {'assembly_backend': 'lagfirst_chunked', 'assembly_precision': 'complex64', 'row_chunk_size': 128, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 16, 'tl_tp_interp_kind': 'linear'}
  chunked_c64_R128_interp32: {'assembly_backend': 'lagfirst_chunked', 'assembly_precision': 'complex64', 'row_chunk_size': 128, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 32, 'tl_tp_interp_kind': 'linear'}
  chunked_c64_R128_interp64: {'assembly_backend': 'lagfirst_chunked', 'assembly_precision': 'complex64', 'row_chunk_size': 128, 'tl_t

In [3]:
# --- Cell 3: comparison and timing helpers ---

from WDM_LISAresponse.code.utils.utils import (
    wdm_mismatch,
    compare_response_signals,
    compute_wdm_cut_info,
    apply_wdm_cut,
)


def rel_l2(test, ref):
    test = np.asarray(test)
    ref = np.asarray(ref)
    num = np.linalg.norm(test - ref)
    den = np.linalg.norm(ref)
    return float(num / den) if den != 0 else np.nan


def max_abs(test, ref):
    test = np.asarray(test)
    ref = np.asarray(ref)
    return float(np.max(np.abs(test - ref)))


def norm_ratio(test, ref):
    test = np.asarray(test)
    ref = np.asarray(ref)
    test_norm = np.linalg.norm(test)
    ref_norm = np.linalg.norm(ref)
    return float(test_norm / ref_norm) if ref_norm != 0 else np.nan


def compare_wdm_arrays(test, ref):
    """
    Compare WDM coefficient arrays using project utilities plus simple diagnostics.
    """
    overlap, mismatch = wdm_mismatch(ref, test)

    return {
        "rel_l2": rel_l2(test, ref),
        "max_abs": max_abs(test, ref),
        "norm_ratio": norm_ratio(test, ref),
        "overlap": float(overlap),
        "mismatch": float(mismatch),
    }


def compare_td_arrays(test, ref):
    """
    Compare time-domain response arrays using project utility.
    """
    d = compare_response_signals(ref, test)

    return {
        "rel_l2": float(d["rel_L2"]),
        "max_abs": max_abs(test, ref),
        "norm_ratio": norm_ratio(test, ref),
        "overlap": float(d["overlap"]),
        "mismatch": float(d["mismatch"]),
    }


def compare_arrays(test, ref, domain="wdm"):
    """
    Generic comparison wrapper.

    domain="wdm" uses wdm_mismatch.
    domain="td" uses compare_response_signals.
    """
    if domain == "td":
        return compare_td_arrays(test, ref)
    return compare_wdm_arrays(test, ref)


def time_call(fn, warm=1, repeat=1):
    for _ in range(max(0, int(warm))):
        _ = fn()

    t0 = time.perf_counter()
    out = None

    for _ in range(max(1, int(repeat))):
        out = fn()

    t1 = time.perf_counter()
    return out, (t1 - t0) / max(1, int(repeat))


def print_table(rows, columns):
    widths = [len(col) for col in columns]

    for row in rows:
        for i, col in enumerate(columns):
            widths[i] = max(widths[i], len(str(row.get(col, ""))))

    header = "  ".join(col.ljust(widths[i]) for i, col in enumerate(columns))
    print(header)
    print("-" * len(header))

    for row in rows:
        print("  ".join(str(row.get(col, "")).ljust(widths[i]) for i, col in enumerate(columns)))


print("Imported project utils and prepared comparison/timing helpers.")

Imported project utils and prepared comparison/timing helpers.


In [4]:
# --- Cell 4: direct target-mode time-shift benchmark ---

direct_backends = selected_backends()

wdm = WDM.WDM.WDM_transform(
    dt=0.5,
    Nf=16,
    N=256,
    q=4,
    calc_m0=True,
)

rng = np.random.default_rng(1234)

w_xi = (
    rng.normal(size=(wdm.Nt, wdm.Nf))
    + 1j * rng.normal(size=(wdm.Nt, wdm.Nf))
)

t_shift = 0.12 * np.sin(
    np.linspace(0.0, 2.0 * np.pi, wdm.Nt)
)

results_direct = {}

for name, kwargs in direct_backends.items():
    print("Running direct backend:", name)

    def run():
        return tsf.wdm_time_shift_variable(
            wdm,
            w_xi,
            t_shift,
            Nf=wdm.Nf,
            L_trunc=3,
            **kwargs,
        )

    out, t_avg = time_call(run, warm=1, repeat=1)

    results_direct[name] = {
        "time_s": t_avg,
        "output": out,
    }

# Prefer faithful chunked reference. If unavailable, use first available backend.
if "chunked_c128_R128" in results_direct:
    ref_name = "chunked_c128_R128"
elif "legacy_c128" in results_direct:
    ref_name = "legacy_c128"
else:
    ref_name = next(iter(results_direct))

ref_out = results_direct[ref_name]["output"]
ref_time = results_direct[ref_name]["time_s"]

rows_direct = []

for name, row in results_direct.items():
    cmp = compare_arrays(row["output"], ref_out, domain="wdm")
    speedup = ref_time / row["time_s"] if row["time_s"] > 0 else np.nan

    rows_direct.append({
        "backend": name,
        "time_s": f"{row['time_s']:.4f}",
        "speedup_vs_ref": f"{speedup:.2f}",
        "rel_l2": f"{cmp['rel_l2']:.3e}",
        "norm_ratio": f"{cmp['norm_ratio']:.6f}",
        "mismatch": f"{cmp['mismatch']:.3e}",
        "max_abs": f"{cmp['max_abs']:.3e}",
    })

print("Reference backend:", ref_name)
print_table(
    rows_direct,
    [
        "backend",
        "time_s",
        "speedup_vs_ref",
        "rel_l2",
        "norm_ratio",
        "mismatch",
        "max_abs",
    ],
)

Running direct backend: chunked_c64_R128_interp4
Running direct backend: chunked_c64_R128_interp8
Running direct backend: chunked_c64_R128_interp16
Running direct backend: chunked_c64_R128_interp32
Running direct backend: chunked_c64_R128_interp64
Reference backend: chunked_c64_R128_interp4
backend                    time_s  speedup_vs_ref  rel_l2     norm_ratio  mismatch   max_abs  
----------------------------------------------------------------------------------------------
chunked_c64_R128_interp4   0.0015  1.00            0.000e+00  1.000000    0.000e+00  0.000e+00
chunked_c64_R128_interp8   0.0010  1.47            8.588e-06  1.000005    2.339e-11  3.681e-05
chunked_c64_R128_interp16  0.0008  1.82            1.011e-05  1.000006    3.209e-11  4.485e-05
chunked_c64_R128_interp32  0.0009  1.59            1.027e-05  1.000006    3.287e-11  4.489e-05
chunked_c64_R128_interp64  0.0009  1.58            1.033e-05  1.000006    3.323e-11  4.521e-05


In [5]:
# --- Cell 5: generate waveform and orbit objects once ---

# Laptop-safe defaults.
# Increase these only after the smoke benchmark is stable.
DURATION_YR = 1.0
dt = 10
Nf = 100
q_wdm = 4
L_trunc_value = 25

alpha = 0.01
tukey_factor = 4

theta = np.pi / 2
phi = 0.0

print("Setting up waveform/orbits...")
print("DURATION_YR:", DURATION_YR)
print("dt:", dt)
print("Nf:", Nf)
print("q_wdm:", q_wdm)
print("L_trunc_value:", L_trunc_value)

td_gen = GenerateEMRIWaveform(
    "FastKerrEccentricEquatorialFlux",
    sum_kwargs=dict(pad_output=True, odd_len=True),
    return_list=True,
)

orbit_data = read_orbit_file("esa-trailing-orbits.h5")
orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
orbits_trailing = InterpolatedOrbits(
    orbit_times,
    orbit_data.positions,
    orbit_data.velocities,
)

t0 = orbit_data.t0 + 0.1 * YRSID_SI

t0 = orbit_data.t0 + 0.1 * YRSID_SI
T_obs = DURATION_YR * YRSID_SI
N = int(np.round(T_obs / dt))

model_kwargs = {
    "td_gen": td_gen,
    "m1": 1e6,
    "m2": 10.0,
    "a": 0.9,
    "p0": 8.5,
    "e0": 0.3,
    "x0": 1.0,
    "dist": 1.0,
    "theta": theta,
    "phi": phi,
    "qK": np.pi / 3,
    "phiK": np.pi / 3,
    "Phi_phi0": np.pi / 3,
    "Phi_theta0": 0.0,
    "Phi_r0": np.pi / 3,
    "eps": 1e-2,
}

window_kwargs = {
    "alpha": alpha,
    "tukey_factor": tukey_factor,
}

print("Generating waveform...")
print("N before padding:", N)

times_pad, hplus_td, hcross_td, pad_info = generate_windowed_padded_waveform(
    waveform_model="emri",
    t0=t0,
    dt=dt,
    N=N,
    model_kwargs=model_kwargs,
    window_function="tukey",
    window_kwargs=window_kwargs,
    wdm_Nf=Nf,
    return_padding_info=True,
)

print("Generated waveform.")
print("times_pad shape:", times_pad.shape)
print("hplus_td shape:", hplus_td.shape)
print("hcross_td shape:", hcross_td.shape)
print("pad_info:", pad_info)

Setting up waveform/orbits...
DURATION_YR: 1.0
dt: 10
Nf: 100
q_wdm: 4
L_trunc_value: 25
Generating waveform...
N before padding: 3155815
Generated waveform.
times_pad shape: (3282200,)
hplus_td shape: (3282200,)
hcross_td shape: (3282200,)
pad_info: {'original_length': 3155815, 'window_pad_left': 63117, 'window_pad_right': 63117, 'window_function': 'tukey', 'wdm_Nf': 100, 'wdm_tail_pad_right': 151, 'wdm_target_length': 3282200, 'wdm_Nt': 32822}


In [6]:
# --- Cell 5b: build TD one-arm reference and transform to WDM ---

print("Computing direct TD one-arm reference...")

TD_NUM_INT_POINTS = 80

def run_td_reference():
    return one_arm_response_td(
        times_pad,
        1,
        2,
        orbits_trailing,
        theta,
        phi,
        hplus_td,
        hcross_td,
        num_int_points=TD_NUM_INT_POINTS,
    )

td_ref_signal, td_ref_time_s = time_call(
    run_td_reference,
    warm=0,
    repeat=1,
)

print("TD reference shape:", td_ref_signal.shape)
print("TD reference runtime_s:", td_ref_time_s)

if not np.all(np.isfinite(td_ref_signal)):
    print("WARNING: TD reference contains non-finite values.")


# Build a WDM transform matching the benchmark configuration.
# This should match the WDM tiling used by one_arm_response_wdm_dT for already padded inputs.
wdm_td_ref = WDM.WDM.WDM_transform(
    dt=dt,
    Nf=Nf,
    N=len(times_pad),
    q=q_wdm,
    calc_m0=True,
    d=4,
    A_frac=0.25,
)

print("TD-reference WDM object:")
print("  Nt:", wdm_td_ref.Nt)
print("  Nf:", wdm_td_ref.Nf)
print("  N:", wdm_td_ref.N)
print("  dT:", wdm_td_ref.dT)

td_ref_wdm = wdm_td_ref.dwt(td_ref_signal)

print("TD reference WDM shape:", td_ref_wdm.shape)

# Conservative science-region cut:
# remove explicit window padding, Tukey taper-affected margin, WDM tail padding, and m=0.
cut_info = compute_wdm_cut_info(
    original_length=N,
    Nf=Nf,
    dt=dt,
    alpha=alpha,
    tukey_factor=tukey_factor,
    window_pad_left=pad_info.get("window_pad_left", None),
    window_pad_right=pad_info.get("window_pad_right", None),
    include_wdm_tail_padding=True,
    include_taper_margin=True,
    wdm_dT=Nf * dt,
    wdm_Nt=td_ref_wdm.shape[0],
    drop_m0=True,
)

td_ref_wdm_cut = apply_wdm_cut(td_ref_wdm, cut_info)

print("cut_info:")
print({
    "time_slice": cut_info["time_slice"],
    "freq_slice": cut_info["freq_slice"],
    "left_cut": cut_info["left_cut"],
    "right_cut": cut_info["right_cut"],
    "trim_m0": cut_info["trim_m0"],
})

print("TD reference WDM cut shape:", td_ref_wdm_cut.shape)

Computing direct TD one-arm reference...
TD reference shape: (3282200,)
TD reference runtime_s: 145.69717175001279
TD-reference WDM object:
  Nt: 32822
  Nf: 100
  N: 3282200
  dT: 1000.0
TD reference WDM shape: (32822, 100)
cut_info:
{'time_slice': slice(789, 32031, None), 'freq_slice': slice(1, None, None), 'left_cut': 789, 'right_cut': 791, 'trim_m0': True}
TD reference WDM cut shape: (31242, 99)


In [9]:
# --- Cell 6: one-arm WDM response benchmark against TD-derived WDM reference ---
# Context-reused version: builds WDM/DWT context once, then runs each backend.

one_arm_backends = selected_backends()

print("Running one-arm benchmark with backends:")
for name, kwargs in one_arm_backends.items():
    print(f"  {name}: {kwargs}")

if not one_arm_backends:
    raise RuntimeError("No active one-arm backends. Check selected_backends().")

# -------------------------------------------------------------------
# Build reusable WDM context once
# -------------------------------------------------------------------

print("\n" + "=" * 80)
print("Building reusable WDM context once...")

sky_basis = build_sky_basis(theta, phi)

def build_one_arm_context():
    return _build_wdm_dT_context(
        times_pad,
        hplus_td,
        hcross_td,
        Nf=Nf,
        d=4,
        q=q_wdm,
        calc_m0=True,
    )

one_arm_context, one_arm_context_time_s = time_call(
    build_one_arm_context,
    warm=0,
    repeat=1,
)

wdm_context = one_arm_context["wdm"]

print("context_build_time_s:", one_arm_context_time_s)
print("WDM context:")
print("  Nt:", wdm_context.Nt)
print("  Nf:", wdm_context.Nf)
print("  N:", wdm_context.N)
print("  dT:", wdm_context.dT)
print("  dF:", wdm_context.dF)

# Optional warm-up with the first backend.
# This prevents first JAX/cache costs from contaminating the first measured backend.
WARM_BACKEND_BEFORE_TIMING = True

if WARM_BACKEND_BEFORE_TIMING:
    first_name = next(iter(one_arm_backends))
    first_kwargs = one_arm_backends[first_name]
    print("\nWarming context-reused response path with backend:", first_name)

    _ = _one_arm_response_wdm_dT_from_context(
        one_arm_context,
        1,
        2,
        orbits_trailing,
        theta,
        phi,
        return_td=False,
        L_trunc=L_trunc_value,
        variable_shift_kwargs=first_kwargs,
        sky_basis=sky_basis,
    )

    print("Warm-up complete.")

# -------------------------------------------------------------------
# Run each backend using the same context
# -------------------------------------------------------------------

results_one_arm = {}

for name, kwargs in one_arm_backends.items():
    print("\n" + "=" * 80)
    print("Running one-arm backend:", name)
    print("kwargs:", kwargs)

    def run():
        return _one_arm_response_wdm_dT_from_context(
            one_arm_context,
            1,
            2,
            orbits_trailing,
            theta,
            phi,
            return_td=False,
            L_trunc=L_trunc_value,
            variable_shift_kwargs=kwargs,
            sky_basis=sky_basis,
        )

    out, t_avg = time_call(run, warm=0, repeat=1)

    if not np.all(np.isfinite(out)):
        print("WARNING: output contains non-finite values.")

    print("output shape:", out.shape)
    print("runtime_s_context_reused:", t_avg)

    results_one_arm[name] = {
        "time_s": t_avg,
        "context_build_time_s": one_arm_context_time_s,
        "output": out,
    }


# -------------------------------------------------------------------
# Main reference: direct TD response transformed to WDM
# -------------------------------------------------------------------

if "td_ref_wdm_cut" not in globals():
    raise RuntimeError("td_ref_wdm_cut is not defined. Run Cell 5b first.")

if "cut_info" not in globals():
    raise RuntimeError("cut_info is not defined. Run Cell 5b first.")

ref_name = "td_reference_wdm"
ref_out_cut = td_ref_wdm_cut
ref_time = td_ref_time_s if "td_ref_time_s" in globals() else np.nan

print("\nMain reference:", ref_name)
print("TD WDM reference cut shape:", ref_out_cut.shape)


# -------------------------------------------------------------------
# Internal faithful WDM reference for backend-to-backend diagnostics
# -------------------------------------------------------------------

# Prefer exact complex128 WDM if available.
if "chunked_c128_R128_exact" in results_one_arm:
    faithful_name = "chunked_c128_R128_exact"
elif "chunked_c128_R128" in results_one_arm:
    faithful_name = "chunked_c128_R128"
elif "legacy_c128_exact" in results_one_arm:
    faithful_name = "legacy_c128_exact"
elif "legacy_c128" in results_one_arm:
    faithful_name = "legacy_c128"
elif "chunked_c128_R128_interp64" in results_one_arm:
    faithful_name = "chunked_c128_R128_interp64"
elif "chunked_c128_R128_interp32" in results_one_arm:
    faithful_name = "chunked_c128_R128_interp32"
elif "chunked_c128_R128_interp16" in results_one_arm:
    faithful_name = "chunked_c128_R128_interp16"
elif "chunked_c128_R128_interp8" in results_one_arm:
    faithful_name = "chunked_c128_R128_interp8"
elif "chunked_c128_R128_interp4" in results_one_arm:
    faithful_name = "chunked_c128_R128_interp4"
else:
    faithful_name = None

faithful_cut = None
faithful_time = np.nan

if faithful_name is not None:
    faithful_full = results_one_arm[faithful_name]["output"]
    faithful_cut = apply_wdm_cut(faithful_full, cut_info)
    faithful_time = results_one_arm[faithful_name]["time_s"]

    print("Internal WDM reference:", faithful_name)
    print("Internal WDM reference full shape:", faithful_full.shape)
    print("Internal WDM reference cut shape:", faithful_cut.shape)

    if "exact" not in faithful_name and "legacy" not in faithful_name:
        print(
            "WARNING: internal WDM reference is not exact. "
            "rel_l2_vs_faithful is only a fast-mode consistency diagnostic."
        )
else:
    print("No internal WDM reference found.")


# -------------------------------------------------------------------
# Build comparison table
# -------------------------------------------------------------------

rows_one_arm = []

for name, row in results_one_arm.items():
    out_full = row["output"]
    out_cut = apply_wdm_cut(out_full, cut_info)

    if out_cut.shape != ref_out_cut.shape:
        raise ValueError(
            f"Cut shape mismatch for {name}: "
            f"out_cut.shape={out_cut.shape}, ref_out_cut.shape={ref_out_cut.shape}"
        )

    # Main science comparison: WDM backend vs direct TD response transformed to WDM.
    cmp_td = compare_arrays(out_cut, ref_out_cut, domain="wdm")

    # Backend-only diagnostic: WDM backend vs faithful/exact WDM backend.
    if faithful_cut is not None:
        if out_cut.shape != faithful_cut.shape:
            raise ValueError(
                f"Faithful cut shape mismatch for {name}: "
                f"out_cut.shape={out_cut.shape}, faithful_cut.shape={faithful_cut.shape}"
            )

        cmp_faithful = compare_arrays(out_cut, faithful_cut, domain="wdm")
        rel_l2_vs_faithful = cmp_faithful["rel_l2"]
        norm_ratio_vs_faithful = cmp_faithful["norm_ratio"]
        overlap_vs_faithful = cmp_faithful["overlap"]
        mismatch_vs_faithful = cmp_faithful["mismatch"]
        max_abs_vs_faithful = cmp_faithful["max_abs"]
    else:
        rel_l2_vs_faithful = np.nan
        norm_ratio_vs_faithful = np.nan
        overlap_vs_faithful = np.nan
        mismatch_vs_faithful = np.nan
        max_abs_vs_faithful = np.nan

    speedup_vs_faithful = (
        faithful_time / row["time_s"]
        if np.isfinite(faithful_time) and row["time_s"] > 0
        else np.nan
    )

    # Useful estimate for a standalone one-shot cost after warm-up:
    # one context build + this context-reused backend call.
    estimated_one_shot_warm_s = one_arm_context_time_s + row["time_s"]

    rows_one_arm.append({
        "backend": name,
        "time_s": f"{row['time_s']:.4f}",
        "context_build_s": f"{one_arm_context_time_s:.4f}",
        "est_one_shot_warm_s": f"{estimated_one_shot_warm_s:.4f}",
        "speedup_vs_faithful": f"{speedup_vs_faithful:.2f}",
        "full_shape": str(out_full.shape),
        "cut_shape": str(out_cut.shape),

        # Main comparison against direct TD response transformed to WDM.
        "rel_l2_vs_td": f"{cmp_td['rel_l2']:.3e}",
        "norm_ratio_vs_td": f"{cmp_td['norm_ratio']:.6f}",
        "overlap_vs_td": f"{cmp_td['overlap']:.12f}",
        "mismatch_vs_td": f"{cmp_td['mismatch']:.3e}",
        "max_abs_vs_td": f"{cmp_td['max_abs']:.3e}",

        # Backend-only comparison against exact/faithful WDM route.
        "rel_l2_vs_faithful": f"{rel_l2_vs_faithful:.3e}",
        "norm_ratio_vs_faithful": f"{norm_ratio_vs_faithful:.6f}",
        "overlap_vs_faithful": f"{overlap_vs_faithful:.12f}",
        "mismatch_vs_faithful": f"{mismatch_vs_faithful:.3e}",
        "max_abs_vs_faithful": f"{max_abs_vs_faithful:.3e}",
    })


print("\nOne-arm benchmark summary on WDM cut region:")
print_table(
    rows_one_arm,
    [
        "backend",
        "time_s",
        "context_build_s",
        "est_one_shot_warm_s",
        "speedup_vs_faithful",
        "full_shape",
        "cut_shape",
        "rel_l2_vs_td",
        "norm_ratio_vs_td",
        "mismatch_vs_td",
        "rel_l2_vs_faithful",
        "norm_ratio_vs_faithful",
        "mismatch_vs_faithful",
        "max_abs_vs_faithful",
    ],
)

Running one-arm benchmark with backends:
  chunked_c64_R128_interp4: {'assembly_backend': 'lagfirst_chunked', 'assembly_precision': 'complex64', 'row_chunk_size': 128, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 4, 'tl_tp_interp_kind': 'linear'}
  chunked_c64_R128_interp8: {'assembly_backend': 'lagfirst_chunked', 'assembly_precision': 'complex64', 'row_chunk_size': 128, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 8, 'tl_tp_interp_kind': 'linear'}
  chunked_c64_R128_interp16: {'assembly_backend': 'lagfirst_chunked', 'assembly_precision': 'complex64', 'row_chunk_size': 128, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 16, 'tl_tp_interp_kind': 'linear'}
  chunked_c64_R128_interp32: {'assembly_backend': 'lagfirst_chunked', 'assembly_precision': 'complex64', 'row_chunk_size': 128, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 32, 'tl_tp_interp_kind': 'linear'}
  chunked_c64_R128_interp64: {'assembly_backend': 'lagfirst_chunked', 'assembly_precision': 'complex64', 'row_chunk_size': 

## Optional TDI Benchmark (Heavy)

Set `RUN_TDI = True` to execute. This is intentionally disabled on laptops by default.

In [ ]:
# --- Optional Cell: TD Gen-2 Michelson TDI reference on a safe short slice ---

RUN_TDI_TD_REFERENCE = True

# Full TD TDI reference is too memory-heavy at multi-million sample length.
# Use a short slice for a smoke/accuracy sanity check.
TDI_TD_REFERENCE_MAX_SAMPLES = 100_000

TDI_TD_NUM_INT_POINTS = 80
TDI_TD_ORDER = 7
TDI_TD_MODE = "zero"

if RUN_TDI_TD_REFERENCE:
    print("Computing TD Gen-2 Michelson TDI reference on a short slice...")
    print("TDI_TD_REFERENCE_MAX_SAMPLES:", TDI_TD_REFERENCE_MAX_SAMPLES)
    print("TDI_TD_NUM_INT_POINTS:", TDI_TD_NUM_INT_POINTS)
    print("TDI_TD_ORDER:", TDI_TD_ORDER)
    print("TDI_TD_MODE:", TDI_TD_MODE)

    n_tdi_ref = min(int(TDI_TD_REFERENCE_MAX_SAMPLES), len(times_pad))

    times_tdi_td = times_pad[:n_tdi_ref]
    hplus_tdi_td = hplus_td[:n_tdi_ref]
    hcross_tdi_td = hcross_td[:n_tdi_ref]

    print("TD TDI slice length:", n_tdi_ref)
    print("TD TDI slice duration days:", (times_tdi_td[-1] - times_tdi_td[0]) / 86400.0)

    tdi_td_shifts_cache = {}

    def run_tdi_td_reference():
        return gen2_michelson_td(
            times_tdi_td,
            hplus_tdi_td,
            hcross_tdi_td,
            theta,
            phi,
            orbits_trailing,
            order=TDI_TD_ORDER,
            mode=TDI_TD_MODE,
            num_int_points=TDI_TD_NUM_INT_POINTS,
            shifts_cache=tdi_td_shifts_cache,
        )

    (X_td, Y_td, Z_td), tdi_td_time_s = time_call(
        run_tdi_td_reference,
        warm=0,
        repeat=1,
    )

    print("TD TDI runtime_s:", tdi_td_time_s)
    print("TD X/Y/Z shapes:", X_td.shape, Y_td.shape, Z_td.shape)

    if not (
        np.all(np.isfinite(X_td))
        and np.all(np.isfinite(Y_td))
        and np.all(np.isfinite(Z_td))
    ):
        print("WARNING: TD TDI reference contains non-finite values.")

    # Build WDM transform for the sliced TD reference.
    # Note: this is a separate short-slice validation WDM grid.
    wdm_tdi_ref = WDM.WDM.WDM_transform(
        dt=dt,
        Nf=Nf,
        N=len(times_tdi_td),
        q=q_wdm,
        calc_m0=True,
        d=4,
        A_frac=0.25,
    )

    print("TDI TD-reference WDM object:")
    print("  Nt:", wdm_tdi_ref.Nt)
    print("  Nf:", wdm_tdi_ref.Nf)
    print("  N:", wdm_tdi_ref.N)

    X_td_wdm = wdm_tdi_ref.dwt(X_td)
    Y_td_wdm = wdm_tdi_ref.dwt(Y_td)
    Z_td_wdm = wdm_tdi_ref.dwt(Z_td)

    print("TD TDI WDM shapes:", X_td_wdm.shape, Y_td_wdm.shape, Z_td_wdm.shape)

    # For the short slice, compare on the whole WDM grid except m=0.
    # This avoids trying to reuse full-run pad_info cuts on a short artificial slice.
    cut_info_tdi = {
        "time_slice": slice(0, X_td_wdm.shape[0]),
        "freq_slice": slice(1, X_td_wdm.shape[1]),
        "left_cut": 0,
        "right_cut": 0,
        "trim_m0": True,
    }

    X_td_wdm_cut = apply_wdm_cut(X_td_wdm, cut_info_tdi)
    Y_td_wdm_cut = apply_wdm_cut(Y_td_wdm, cut_info_tdi)
    Z_td_wdm_cut = apply_wdm_cut(Z_td_wdm, cut_info_tdi)

    tdi_td_ref_concat_cut = np.concatenate([
        np.asarray(X_td_wdm_cut).ravel(),
        np.asarray(Y_td_wdm_cut).ravel(),
        np.asarray(Z_td_wdm_cut).ravel(),
    ])

    print("cut_info_tdi:")
    print({
        "time_slice": cut_info_tdi["time_slice"],
        "freq_slice": cut_info_tdi["freq_slice"],
        "left_cut": cut_info_tdi["left_cut"],
        "right_cut": cut_info_tdi["right_cut"],
        "trim_m0": cut_info_tdi["trim_m0"],
    })

    print("TD TDI WDM cut shapes:", X_td_wdm_cut.shape, Y_td_wdm_cut.shape, Z_td_wdm_cut.shape)

else:
    print("RUN_TDI_TD_REFERENCE is False; skipping TD TDI reference.")

In [ ]:
# --- Optional Cell: full-size TDI WDM benchmark, fast modes only ---

RUN_TDI = True

# Full-size TDI should not require TD TDI reference on laptop.
# Compare fast WDM modes internally and record runtime/stability.
TDI_BACKENDS = {
    "c64_interp16": {
        "assembly_backend": "lagfirst_chunked",
        "assembly_precision": "complex64",
        "row_chunk_size": 128,
        "tl_tp_mode": "interp",
        "tl_tp_interp_points": 16,
        "tl_tp_interp_kind": "linear",
    },
    "c64_interp32": {
        "assembly_backend": "lagfirst_chunked",
        "assembly_precision": "complex64",
        "row_chunk_size": 128,
        "tl_tp_mode": "interp",
        "tl_tp_interp_points": 32,
        "tl_tp_interp_kind": "linear",
    },
    # Use this mainly for Nf=50 or if you want the safest tested interpolation mode.
    "c64_interp64": {
        "assembly_backend": "lagfirst_chunked",
        "assembly_precision": "complex64",
        "row_chunk_size": 128,
        "tl_tp_mode": "interp",
        "tl_tp_interp_points": 64,
        "tl_tp_interp_kind": "linear",
    },
}

rows_tdi = []
results_tdi = {}

if RUN_TDI:
    print("Running full-size TDI WDM benchmark with fast backends:")
    print("Nf:", Nf)
    print("q_wdm:", q_wdm)
    print("L_trunc_value:", L_trunc_value)

    for name, kwargs in TDI_BACKENDS.items():
        print(f"  {name}: {kwargs}")

    # Use shared caches across backend runs only for geometry/shifts where safe.
    # The one-arm cache must NOT be shared across different variable_shift_kwargs,
    # because precompute_one_arm_wdm_dT_links can cache by shift kwargs internally,
    # but keeping it separate per backend is safest in notebook testing.
    shared_tdi_wdm_shifts_cache = {}

    for name, kwargs in TDI_BACKENDS.items():
        print("\n" + "=" * 80)
        print("Running TDI WDM backend:", name)
        print("kwargs:", kwargs)

        tdi_wdm_one_arm_cache = {}

        def run():
            X_wdm, Y_wdm, Z_wdm = gen2_michelson_wdm(
                times_pad,
                hplus_td,
                hcross_td,
                theta,
                phi,
                orbits_trailing,
                Nf=Nf,
                shifts_cache=shared_tdi_wdm_shifts_cache,
                one_arm_cache=tdi_wdm_one_arm_cache,
                L_trunc=L_trunc_value,
                variable_shift_kwargs=kwargs,
                q=q_wdm,
            )
            return X_wdm, Y_wdm, Z_wdm

        # No warm run: Gen-2 TDI is expensive.
        out, t_avg = time_call(run, warm=0, repeat=1)

        X_wdm, Y_wdm, Z_wdm = out

        print("X/Y/Z shapes:", X_wdm.shape, Y_wdm.shape, Z_wdm.shape)
        print("runtime_s:", t_avg)

        finite = (
            np.all(np.isfinite(X_wdm))
            and np.all(np.isfinite(Y_wdm))
            and np.all(np.isfinite(Z_wdm))
        )

        if not finite:
            print("WARNING: TDI WDM output contains non-finite values.")

        results_tdi[name] = {
            "time_s": t_avg,
            "output": out,
            "finite": finite,
        }

    # Internal fast reference: choose the highest interpolation point available.
    # This is not exact; it is only a fast-mode consistency reference.
    if "c64_interp64" in results_tdi:
        ref_name_tdi_internal = "c64_interp64"
    elif "c64_interp32" in results_tdi:
        ref_name_tdi_internal = "c64_interp32"
    else:
        ref_name_tdi_internal = next(iter(results_tdi))

    ref_X, ref_Y, ref_Z = results_tdi[ref_name_tdi_internal]["output"]
    ref_time_internal = results_tdi[ref_name_tdi_internal]["time_s"]

    # Build a WDM cut for the full-size TDI outputs.
    # This mirrors the one-arm cut: remove padding/taper margin and m=0.
    cut_info_tdi_wdm = compute_wdm_cut_info(
        original_length=N,
        Nf=Nf,
        dt=dt,
        alpha=alpha,
        tukey_factor=tukey_factor,
        window_pad_left=pad_info.get("window_pad_left", None),
        window_pad_right=pad_info.get("window_pad_right", None),
        include_wdm_tail_padding=True,
        include_taper_margin=True,
        wdm_dT=Nf * dt,
        wdm_Nt=ref_X.shape[0],
        drop_m0=True,
    )

    print("\ncut_info_tdi_wdm:")
    print({
        "time_slice": cut_info_tdi_wdm["time_slice"],
        "freq_slice": cut_info_tdi_wdm["freq_slice"],
        "left_cut": cut_info_tdi_wdm["left_cut"],
        "right_cut": cut_info_tdi_wdm["right_cut"],
        "trim_m0": cut_info_tdi_wdm["trim_m0"],
    })

    ref_X_cut = apply_wdm_cut(ref_X, cut_info_tdi_wdm)
    ref_Y_cut = apply_wdm_cut(ref_Y, cut_info_tdi_wdm)
    ref_Z_cut = apply_wdm_cut(ref_Z, cut_info_tdi_wdm)

    ref_concat_internal = np.concatenate([
        np.asarray(ref_X_cut).ravel(),
        np.asarray(ref_Y_cut).ravel(),
        np.asarray(ref_Z_cut).ravel(),
    ])

    print("Internal fast TDI reference:", ref_name_tdi_internal)
    print("Internal cut X/Y/Z shapes:", ref_X_cut.shape, ref_Y_cut.shape, ref_Z_cut.shape)

    for name, row in results_tdi.items():
        X, Y, Z = row["output"]

        X_cut = apply_wdm_cut(X, cut_info_tdi_wdm)
        Y_cut = apply_wdm_cut(Y, cut_info_tdi_wdm)
        Z_cut = apply_wdm_cut(Z, cut_info_tdi_wdm)

        test_concat = np.concatenate([
            np.asarray(X_cut).ravel(),
            np.asarray(Y_cut).ravel(),
            np.asarray(Z_cut).ravel(),
        ])

        if test_concat.shape != ref_concat_internal.shape:
            raise ValueError(
                f"Internal concat shape mismatch for {name}: "
                f"{test_concat.shape} vs {ref_concat_internal.shape}"
            )

        cmp_all = compare_arrays(test_concat, ref_concat_internal, domain="wdm")
        cmp_X = compare_arrays(X_cut, ref_X_cut, domain="wdm")
        cmp_Y = compare_arrays(Y_cut, ref_Y_cut, domain="wdm")
        cmp_Z = compare_arrays(Z_cut, ref_Z_cut, domain="wdm")

        speedup_vs_internal = (
            ref_time_internal / row["time_s"]
            if row["time_s"] > 0
            else np.nan
        )

        rows_tdi.append({
            "backend": name,
            "time_s": f"{row['time_s']:.4f}",
            "speedup_vs_internal": f"{speedup_vs_internal:.2f}",
            "finite": str(row["finite"]),
            "full_shape_X": str(X.shape),
            "cut_shape_X": str(X_cut.shape),

            # Fast-mode consistency only. This is NOT exact unless the reference is exact.
            "rel_l2_all_vs_internal": f"{cmp_all['rel_l2']:.3e}",
            "mismatch_all_vs_internal": f"{cmp_all['mismatch']:.3e}",
            "norm_ratio_all_vs_internal": f"{cmp_all['norm_ratio']:.6f}",

            "rel_l2_X_vs_internal": f"{cmp_X['rel_l2']:.3e}",
            "rel_l2_Y_vs_internal": f"{cmp_Y['rel_l2']:.3e}",
            "rel_l2_Z_vs_internal": f"{cmp_Z['rel_l2']:.3e}",

            "mismatch_X_vs_internal": f"{cmp_X['mismatch']:.3e}",
            "mismatch_Y_vs_internal": f"{cmp_Y['mismatch']:.3e}",
            "mismatch_Z_vs_internal": f"{cmp_Z['mismatch']:.3e}",
        })

    print("\nTDI WDM fast-mode consistency summary on WDM cut region:")
    print("Internal reference is fast mode only, not exact:", ref_name_tdi_internal)

    print_table(
        rows_tdi,
        [
            "backend",
            "time_s",
            "speedup_vs_internal",
            "finite",
            "full_shape_X",
            "cut_shape_X",
            "rel_l2_all_vs_internal",
            "mismatch_all_vs_internal",
            "norm_ratio_all_vs_internal",
            "rel_l2_X_vs_internal",
            "rel_l2_Y_vs_internal",
            "rel_l2_Z_vs_internal",
            "mismatch_X_vs_internal",
            "mismatch_Y_vs_internal",
            "mismatch_Z_vs_internal",
        ],
    )

else:
    print("RUN_TDI is False; skipping TDI WDM benchmark.")

In [ ]:
# --- Save benchmark results ---

timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
out_dir = os.path.join(os.getcwd(), "benchmark_results")
os.makedirs(out_dir, exist_ok=True)

tag_parts = []

if "DURATION_YR" in globals():
    tag_parts.append(f"{float(DURATION_YR):.3f}yr")

if "Nf" in globals():
    tag_parts.append(f"Nf{int(Nf)}")

if "L_trunc_value" in globals():
    tag_parts.append(f"L{int(L_trunc_value)}")

tag = "_".join(tag_parts) if tag_parts else "run"


def write_csv(name, rows):
    path = os.path.join(out_dir, f"{name}_{tag}_{timestamp}.csv")

    if not rows:
        print("No rows to save for", name)
        return path

    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    print("Wrote", path)
    return path


if "rows_direct" in globals():
    write_csv("direct_shift", rows_direct)
else:
    print("rows_direct not found; skipping direct_shift save.")

if "rows_one_arm" in globals():
    write_csv("one_arm", rows_one_arm)
else:
    print("rows_one_arm not found; skipping one_arm save.")

if "rows_tdi" in globals() and rows_tdi:
    write_csv("tdi", rows_tdi)
else:
    print("No TDI rows found; skipping TDI save.")